# Day 6 — Notebook 01c: What Do `0` and `1` Mean in `churned`?

In the StreamPulse data, the answer column is called **`churned`**. It only has two values: **0** and **1**.

## ⚠️ The Most Common Confusion

Many people think:
> ❌ "0 = cancel, 1 = start the subscription"

**That is WRONG.** The correct meaning is:

| Value | Meaning | Simple words |
|---|---|---|
| **`churned = 0`** | Customer **STAYED** | Still subscribed ✅ — kept paying |
| **`churned = 1`** | Customer **CANCELLED** | Left / stopped the subscription ❌ |

### 🔑 Important: nobody in this column is "starting" a subscription!
**Every customer in the data already HAS a subscription.** The only question is:

> *"Next month — will this customer **stay** or **leave**?"*

- They **stay** → `0`
- They **leave** → `1`

**Action:** Click **Run → Run All Cells**.

## 1. What Does "Churn" Mean?

**Churn = a customer leaving / cancelling.**

Think of a bucket of water 🪣 with a small hole:
- Water in the bucket = paying customers
- Water leaking out of the hole = **churn** (customers who cancel)

So the column name answers one question:

> **"Did this customer churn (cancel)?"**
> - **No** → `0`
> - **Yes** → `1`

That's why `1` means **cancelled** — `1` is just a short way to write **"Yes, they churned."**

## 2. A Customer's Journey (Timeline)

```text
  SIGN UP            USING THE SERVICE              NEXT MONTH
     │                     │                            │
     ●─────────────────────●────────────────────────────●
  Starts the          Logs in, watches,           Did they cancel?
  subscription        pays every month
                                                   ┌────┴────┐
  (NOT in our                                      │         │
   churned column)                               STAYED   CANCELLED
                                                 churned  churned
                                                  = 0      = 1
```

**Starting a subscription happens at the very beginning** — it's the reason the customer is in our data at all.
The `churned` column only records **what happened at the end**: stay (0) or leave (1).

## 3. See It in the Real StreamPulse Data

In [ ]:
import pandas as pd

customers = pd.read_csv("data/customers_raw.csv")

# Add a plain-English column next to the 0/1 so it's crystal clear
customers["what_it_means"] = customers["churned"].map({
    0: "STAYED (still subscribed)",
    1: "CANCELLED (left)",
})

customers[["customer_id", "plan_tier", "tenure_months", "monthly_charges", "churned", "what_it_means"]].head(10)

Notice `tenure_months` — **every customer has been subscribed for some months already**. They all *started* in the past.
The `churned` column only tells us whether they **stayed (0)** or **cancelled (1)**.

In [ ]:
counts = customers["churned"].value_counts().sort_index()

print("How many customers in each group:\n")
print(f"  churned = 0  ->  STAYED     : {counts[0]:>4} customers ({counts[0] / counts.sum():.1%})")
print(f"  churned = 1  ->  CANCELLED  : {counts[1]:>4} customers ({counts[1] / counts.sum():.1%})")

## 4. Look at One Customer from Each Group

In [ ]:
stayed = customers[customers["churned"] == 0].iloc[0]
cancelled = customers[customers["churned"] == 1].iloc[0]

print("Customer who STAYED (churned = 0)")
print(f"  ID: {stayed['customer_id']} | subscribed for {stayed['tenure_months']:.0f} months | plan: {stayed['plan_tier']}")
print("  -> Started in the past, and is STILL paying. ✅\n")

print("Customer who CANCELLED (churned = 1)")
print(f"  ID: {cancelled['customer_id']} | subscribed for {cancelled['tenure_months']:.0f} months | plan: {cancelled['plan_tier']}")
print("  -> Started in the past, then LEFT. ❌")

## 5. Why Is "Cancel" the `1` and Not the `0`?

In Machine Learning, **`1` = the thing we are trying to FIND** (called the *positive class*).

StreamPulse wants to **find customers who are about to cancel**, so they can send them a discount and keep them.
So "cancelled" gets the `1`.

Same idea in other problems:

| Problem | `1` means (what we're looking for) | `0` means |
|---|---|---|
| **Churn (ours)** | Customer **cancels** | Customer stays |
| Spam filter | Email **is spam** | Normal email |
| Fraud detection | Transaction **is fraud** | Normal transaction |
| Medical test | Patient **has the disease** | Healthy |

> ⚠️ "Positive" does **NOT** mean "good"! Cancelling is bad for the business — it's called "positive" only because it's the thing we're searching for.

## 6. What the Model's Prediction Means

In [ ]:
predictions = pd.DataFrame({
    "customer":          ["Customer A", "Customer B"],
    "model_predicts":    [1, 0],
    "churn_probability": ["92%", "4%"],
})
predictions["meaning"] = predictions["model_predicts"].map({
    1: "Likely to CANCEL -> send a discount offer!",
    0: "Likely to STAY -> no action needed",
})
predictions

- **Prediction `1`** → "This customer will probably **cancel**" → the business should act.
- **Prediction `0`** → "This customer will probably **stay**" → all good.
- **Churn probability** → how sure the model is that they will **cancel** (higher = more risk).

## Remember 🧠

| | `churned = 0` | `churned = 1` |
|---|---|---|
| **Meaning** | **Stayed** | **Cancelled** |
| **Still subscribed?** | ✅ Yes | ❌ No |
| **Good for business?** | 👍 Yes | 👎 No |
| **What we try to predict/find** | — | ✅ This one! |

> **`churned` asks "Did they leave?" → 0 = No (stayed), 1 = Yes (cancelled).**
> **Starting a subscription is NOT in this column — everyone already started.**